# 7장 — 차이가 진짜일까: 짝지은 비교와 p값 ⚖️

6장의 숙제(구간이 겹친 두 모델)를 해결합니다. 열쇠는 **같은 시험지로 맞대결**:

- 둘 다 맞음 / 둘 다 틀림 → 우열 정보 **0** (문항 난이도 이야기)
- **불일치 문항**(A만/B만 맞음)만이 증거. 실력이 같다면 50:50 동전이어야 한다
- 판정: 부호 검정(맥니마) **z = (x−y)/√(x+y)**, p값(3부 9장 복습!)

선수는 워밍업의 그 둘: 진짜 실력 A 72% vs B 70% (신의 시점).

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 여기서 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch7')

## 1. 성장하는 시험: 신의 시점 문항 구조

문항별 확률: 둘 다 맞음 66% / A만 6% / B만 4% / 둘 다 틀림 24% → A 72%, B 70% (2%p 차).
8,000문항 시험(웹 랩과 같은 mulberry32 시드 777)의 앞 n문항으로 두 채점 방식을 대결시킵니다.

In [ ]:
import math

M32 = 0xFFFFFFFF

def imul(a, b):
    return (a * b) & M32

def mulberry32(seed):
    a = seed & M32
    def rand():
        nonlocal a
        a = (a + 0x6D2B79F5) & M32
        t = a
        t = imul(t ^ (t >> 15), t | 1) & M32
        t = (t ^ ((t + imul(t ^ (t >> 7), t | 61)) & M32)) & M32
        return (t ^ (t >> 14)) / 4294967296
    return rand

def phi(z):
    """표준정규 CDF — A&S 7.1.26 근사 (웹 랩과 동일 구현)."""
    x = abs(z) / math.sqrt(2)
    t = 1 / (1 + 0.3275911 * x)
    y = 1 - (((((1.061405429 * t - 1.453152027) * t) + 1.421413741) * t - 0.284496736) * t + 0.254829592) * t * math.exp(-x * x)
    return 0.5 * (1 + y) if z >= 0 else 0.5 * (1 - y)

def p_two(z):
    return 2 * (1 - phi(abs(z)))

assert abs(phi(1.96) - 0.975002) < 1e-6
assert abs(p_two(2) - 0.045500) < 1e-6

# 문항 유형 스트림: 0 둘다맞음 / 1 A만 / 2 B만 / 3 둘다틀림
rand = mulberry32(777)
CAT = []
for _ in range(8000):
    u = rand()
    CAT.append(0 if u < 0.66 else 1 if u < 0.72 else 2 if u < 0.76 else 3)

def stats(n):
    both = sum(1 for c in CAT[:n] if c == 0)
    x = sum(1 for c in CAT[:n] if c == 1)
    y = sum(1 for c in CAT[:n] if c == 2)
    a, b = (both + x) / n, (both + y) / n
    se_u = math.sqrt(a * (1 - a) / n + b * (1 - b) / n)
    z_u = (a - b) / se_u
    z_p = (x - y) / math.sqrt(x + y) if x + y > 0 else 0.0
    return a, b, x, y, z_u, p_two(z_u), z_p, p_two(z_p)

print('    n |  A점수  B점수 | A만  B만 | 따로 z(p)     | 짝지어 z(p)')
print('-' * 72)
for n in [250, 500, 1000, 2000, 4000, 8000]:
    a, b, x, y, zu, pu, zp, pp = stats(n)
    print(f'{n:>5} | {a*100:5.1f}  {b*100:5.1f} | {x:>4} {y:>4} | {zu:5.2f} ({pu:.3f}) | {zp:5.2f} ({pp:.4f})')

# 핵심 수치 못박기 — 같은 데이터, 갈리는 판정
a, b, x, y, zu, pu, zp, pp = stats(1000)
assert (x, y) == (57, 37)
assert round(pu, 3) == 0.324 and round(pp, 4) == 0.0391, 'n=1,000: 따로 판정불가 vs 짝지어 유의'
a, b, x, y, zu, pu, zp, pp = stats(500)
assert x == y == 26, 'n=500은 정확히 동점(p=1) — 증거 없음의 정직한 순간'
print()
print('✔ n=1,000: 따로 p=0.324(불가) vs 짝지어 p=0.039(유의). 채점 방식이 판정을 가른다')

## 2. 왜 짝지은 비교가 이기나: 필요 표본의 이론값

따로 보기의 분산에는 문항 난이도 운이 통째로 들어가고, 짝지은 비교는 이 운을 소거합니다.
2%p 차이를 z=1.96으로 가리는 데 필요한 문항 수를 비교하면:

In [ ]:
var_unpaired = 0.72 * 0.28 + 0.70 * 0.30          # 두 비율 분산 합
var_paired = 0.06 + 0.04 - 0.02 ** 2               # 불일치율 − 차이²
n_unpaired = 1.96 ** 2 * var_unpaired / 0.02 ** 2
n_paired = 1.96 ** 2 * var_paired / 0.02 ** 2
print(f'따로 보기: ≈ {n_unpaired:,.0f}문항')
print(f'짝지어 보기: ≈ {n_paired:,.0f}문항  → 절약 {n_unpaired/n_paired:.1f}배')
assert round(n_unpaired) == 3953 and round(n_paired) == 957
print()
print('✔ 절약 배율은 두 모델이 비슷할수록(불일치가 적을수록) 커진다. 95%+ 일치면 10배 이상')

## 3. 그림: 문항 수에 따른 두 방식의 p값

In [ ]:
import matplotlib.pyplot as plt

try:
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['axes.unicode_minus'] = False
except Exception:
    pass

ns = [250, 500, 1000, 2000, 4000, 8000]
pus = [stats(n)[5] for n in ns]  # 따로 보기 p값
pps = [stats(n)[7] for n in ns]  # 짝지어 보기 p값

plt.figure(figsize=(8, 3.8))
plt.semilogy(ns, pus, 'o-', color='#94a3b8', label='따로 보기 (두 비율)')
plt.semilogy(ns, pps, 'o-', color='#4f46e5', label='짝지어 보기 (부호 검정)')
plt.axhline(0.05, color='#e11d48', ls='--', lw=1, label='α = 0.05')
plt.xlabel('문항 수 n'); plt.ylabel('p값 (로그 축)')
plt.title('같은 데이터, 두 채점: 짝지은 비교가 4배 먼저 유의선을 뚫는다')
plt.legend()
plt.tight_layout()
plt.show()
print('짝지은 비교는 n≈1,000에서, 따로는 n≈4,000에서 0.05 아래로. 이론값(957 vs 3,953)과 일치')

## 정리: 오늘 가져갈 세 문장

1. 우열의 증거는 **불일치 문항**뿐. 실력이 같다면 50:50 동전이어야 한다.
2. 판정: **z = (x−y)/√(x+y)**, p값(3부 9장의 그 정의). z>1.96이면 유의(p<0.05).
3. **같은 문항으로 짝지은 비교**는 공통 소음(난이도 운)을 소거해 필요 표본을 몇 배로 줄인다.
   "A가 이겼다" 주장엔 "같은 문항이었나? p값은?"을 물을 것. (+유의성≠크기, 다중 비교 주의)

다음 **8장, 기출 유출(오염)**: 판정 도구가 완벽해도 시험 자체가 썩으면 소용없다.